<a href="https://colab.research.google.com/github/vituhaa/Multimodal-Reasoning-for-STEM/blob/main/Multimodal_Reasoning_for_STEM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Qwen/Qwen3-VL-2B-Instruct

Qwen/Qwen3-VL-4B-Instruct

HuggingFaceTB/SmolVLM-256M Instruct

google/t5gemma-2-270m-270m

"HuggingFaceTB/SmolVLM-256M Instruct" model was selected to fine-tune.

Photo example from  [linxy/LaTeX_OCR dataset](https://huggingface.co/datasets/linxy/LaTeX_OCR/viewer/human_handwrite?views%5B%5D=human_handwrite_train):

sinx - siny - sin(x - y)

Zero-shot inference

In [ ]:
! nvidia-smi

In [ ]:
! pip install -q torch torchvision
! pip install -q transformers accelerate
! pip install --upgrade pip wheel setuptools
! pip install pillow
! MAX_JOBS=64 pip install "flash-attn<2.0.0" --no-build-isolation

In [ ]:
import torch
import requests
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from transformers.image_utils import load_image
from io import BytesIO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

image_path = "/content/formula.jpg"
image = load_image(image_path)

model_name = "HuggingFaceTB/SmolVLM-256M-Instruct"
processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-256M-Instruct")
model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    _attn_implementation="sdpa"
).to(DEVICE)

input_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this hand-written mathematical formula into LaTex format."}
        ]
    }
]

prompt = processor.apply_chat_template(input_messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors="pt")
inputs = inputs.to(DEVICE)

generated_ids = model.generate(**inputs, max_new_tokens=500)
generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)

print(generated_texts[0])

One-shot inference

In [ ]:
example_image_path = "/content/example.jpg"
inference_image_path = "/content/inference.jpg"
example_image = load_image(example_image_path)
inference_image = load_image(inference_image_path)

model_name = "HuggingFaceTB/SmolVLM-256M-Instruct"
processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-256M-Instruct")
model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    _attn_implementation="sdpa"
).to(DEVICE)

input_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this hand-written mathematical formula into LaTex format."}
        ]
    },
    {
        "role": "assistant",
        "content": [
            {"type": "text", "text": "\log z = \log r + i ( \theta + 2 n \pi )"}
        ]
    },
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this hand-written mathematical formula into LaTex format."}
        ]
    }
]

prompt = processor.apply_chat_template(input_messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[example_image, inference_image], return_tensors="pt")
inputs = inputs.to(DEVICE)

generated_ids = model.generate(**inputs, max_new_tokens=500)
generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)

print(*generated_texts)